# Keyword Dataset — Exploratory Analysis

**Source file:** `data/keywords/keyword_dataset_2026-06.csv`  
**Pipeline:** `analysis/build_keyword_dataset.py`  
**Data sources:** global_journeys · inside_australia · australia_com · discover_tasmania · queensland_com · sydney_com

**Sections**
1. Load & Overview
2. Coverage Metrics
3. Keyword Count Distribution
4. Top Place Tags
5. Top Activity Tags
6. Theme Distribution
7. Source Breakdown
8. State Distribution
9. Best & Worst Covered Tours
10. Theme Co-occurrence

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from collections import Counter
from pathlib import Path
from itertools import combinations

DATA_PATH = Path("..") / "data" / "keywords" / "keyword_dataset_2026-06.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head(3)

## 1. Dataset Overview

In [ ]:
print("=== Basic Info ===")
print(f"Total rows (tour-days)  : {len(df):,}")
print(f"Unique tours            : {df['tour_url'].nunique():,}")
print(f"Unique tour names       : {df['tour_name'].nunique():,}")
print(f"Data sources            : {sorted(df['source'].unique())}")
print(f"States covered          : {sorted(df['state'].dropna().unique())}")
print(f"Date range (scraped)    : {df['scrape_date'].min()} → {df['scrape_date'].max()}")
print()
print("=== Null Counts ===")
print(df[['activity','price','city','place_tags','activity_tags','theme_tags']].isnull().sum().to_string())

## 2. Coverage Metrics

In [ ]:
total = len(df)
metrics = {
    "Has activity text"   : df["has_content"].sum(),
    "Has place_tags"      : df["place_tags"].notna().sum(),
    "Has activity_tags"   : df["activity_tags"].notna().sum(),
    "Has theme_tags"      : df["theme_tags"].notna().sum(),
    "Has any keyword"     : (df["keyword_count"] > 0).sum(),
}

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(list(metrics.keys()), list(metrics.values()), color="steelblue", edgecolor="white")
for bar, val in zip(bars, metrics.values()):
    pct = val / total * 100
    ax.text(bar.get_width() + 4, bar.get_y() + bar.get_height() / 2,
            f"{val:,}  ({pct:.0f}%)", va="center", fontsize=9)
ax.set_xlim(0, total * 1.25)
ax.set_xlabel("Row count")
ax.set_title(f"Tag Coverage  (n = {total:,} tour-days)")
ax.axvline(total, color="grey", linewidth=0.8, linestyle="--", label=f"Total rows ({total:,})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 3. Keyword Count Distribution

In [ ]:
kw_counts = df["keyword_count"]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram
axes[0].hist(kw_counts[kw_counts > 0], bins=30, color="steelblue", edgecolor="white")
axes[0].set_xlabel("Keywords per row")
axes[0].set_ylabel("Row count")
axes[0].set_title("Distribution (rows with ≥1 keyword)")

# Summary stats
stats = kw_counts.describe().rename({
    "count": "total rows", "mean": "mean", "std": "std dev",
    "min": "min", "25%": "25th pct", "50%": "median",
    "75%": "75th pct", "max": "max"
})
axes[1].axis("off")
table_data = [[k, f"{v:.1f}"] for k, v in stats.items()]
tbl = axes[1].table(cellText=table_data, colLabels=["Metric", "Value"],
                    loc="center", cellLoc="left")
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.4, 1.6)
axes[1].set_title("Keyword Count Stats")

plt.tight_layout()
plt.show()

## 4. Top Place Tags

In [ ]:
def flatten_tags(series, sep="; "):
    tags = []
    for cell in series.dropna():
        for t in str(cell).split(sep):
            t = t.strip()
            if t:
                tags.append(t)
    return tags

place_freq = Counter(flatten_tags(df["place_tags"]))
top_places = place_freq.most_common(25)
labels, values = zip(*top_places)

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(labels[::-1], values[::-1], color="#2196F3", edgecolor="white")
for bar, val in zip(bars, values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            str(val), va="center", fontsize=8)
ax.set_xlabel("Frequency (tour-days)")
ax.set_title("Top 25 Place Tags")
plt.tight_layout()
plt.show()

print(f"Total unique place tags : {len(place_freq):,}")
print(f"Tags appearing once     : {sum(1 for v in place_freq.values() if v == 1):,}")

## 5. Top Activity Tags

In [ ]:
act_freq = Counter(flatten_tags(df["activity_tags"]))
top_acts = act_freq.most_common(20)

if top_acts:
    labels, values = zip(*top_acts)
    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(labels[::-1], values[::-1], color="#4CAF50", edgecolor="white")
    for bar, val in zip(bars, values[::-1]):
        ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height() / 2,
                str(val), va="center", fontsize=8)
    ax.set_xlabel("Frequency")
    ax.set_title("Top 20 Activity Tags")
    plt.tight_layout()
    plt.show()
else:
    print("No activity tags found.")

print(f"Total unique activity tags : {len(act_freq):,}")
print(f"Rows with activity tags    : {df['activity_tags'].notna().sum():,}")

## 6. Theme Distribution

In [ ]:
theme_freq = Counter(flatten_tags(df["theme_tags"]))
themes_sorted = sorted(theme_freq.items(), key=lambda x: -x[1])
t_labels, t_values = zip(*themes_sorted)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
axes[0].barh(t_labels[::-1], t_values[::-1], color="#FF9800", edgecolor="white")
for i, (lab, val) in enumerate(zip(t_labels[::-1], t_values[::-1])):
    axes[0].text(val + 1, i, str(val), va="center", fontsize=8)
axes[0].set_xlabel("Frequency")
axes[0].set_title("Theme Tag Frequency")

# Pie chart
axes[1].pie(t_values, labels=t_labels, autopct="%1.0f%%", startangle=140,
            colors=plt.cm.Set3.colors[:len(t_labels)])
axes[1].set_title("Theme Share")

plt.tight_layout()
plt.show()

## 7. Source Breakdown

In [ ]:
# Unique tours per source
tours_per_source = df.drop_duplicates("tour_url").groupby("source").size().sort_values(ascending=False)
days_per_source  = df.groupby("source").size().sort_values(ascending=False)
kw_per_source    = df.groupby("source")["keyword_count"].mean().round(1)

summary = pd.DataFrame({
    "Unique tours": tours_per_source,
    "Tour-days"   : days_per_source,
    "Avg keywords": kw_per_source,
}).fillna(0)

print(summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
summary["Unique tours"].plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white", rot=15)
axes[0].set_title("Unique Tours per Source")
axes[0].set_ylabel("Count")

summary["Avg keywords"].plot(kind="bar", ax=axes[1], color="#FF9800", edgecolor="white", rot=15)
axes[1].set_title("Avg Keywords per Tour-Day")
axes[1].set_ylabel("Avg keyword count")

plt.tight_layout()
plt.show()

## 8. State Distribution

In [ ]:
state_days  = df.groupby("state").size().sort_values(ascending=False)
state_tours = df.drop_duplicates("tour_url").groupby("state").size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
state_days.plot(kind="bar", ax=axes[0], color="#673AB7", edgecolor="white", rot=0)
axes[0].set_title("Tour-Days per State")
axes[0].set_ylabel("Count")

state_tours.plot(kind="bar", ax=axes[1], color="#009688", edgecolor="white", rot=0)
axes[1].set_title("Unique Tours per State")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()

## 9. Best & Worst Covered Tours

In [ ]:
tour_kw = (
    df.groupby(["tour_name", "tour_url", "source"])
    .agg(total_days=("day_number", "max"),
         avg_kw=("keyword_count", "mean"),
         days_with_kw=("keyword_count", lambda x: (x > 0).sum()),
         total_day_rows=("keyword_count", "count"))
    .reset_index()
)
tour_kw["pct_days_with_kw"] = (tour_kw["days_with_kw"] / tour_kw["total_day_rows"] * 100).round(1)
tour_kw["avg_kw"] = tour_kw["avg_kw"].round(1)

print("--- Top 10 best-covered tours (highest avg keywords/day) ---")
print(tour_kw.nlargest(10, "avg_kw")[["tour_name", "source", "total_days", "avg_kw", "pct_days_with_kw"]].to_string(index=False))

print()
print("--- Top 10 worst-covered tours (0 or fewest keywords) ---")
print(tour_kw.nsmallest(10, "avg_kw")[["tour_name", "source", "total_days", "avg_kw", "pct_days_with_kw"]].to_string(index=False))

## 10. Theme Co-occurrence

Which themes tend to appear together on the same tour-day?

In [ ]:
cooccur = Counter()
for cell in df["theme_tags"].dropna():
    themes = sorted(t.strip() for t in str(cell).split("; ") if t.strip())
    for pair in combinations(themes, 2):
        cooccur[pair] += 1

top_pairs = cooccur.most_common(15)
if top_pairs:
    pair_labels = [f"{a}\n+ {b}" for (a, b), _ in top_pairs]
    pair_values = [v for _, v in top_pairs]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(pair_labels[::-1], pair_values[::-1], color="#E91E63", edgecolor="white")
    for i, val in enumerate(pair_values[::-1]):
        ax.text(val + 0.3, i, str(val), va="center", fontsize=8)
    ax.set_xlabel("Co-occurrence count")
    ax.set_title("Top 15 Theme Pairs (co-occurring on the same tour-day)")
    plt.tight_layout()
    plt.show()
else:
    print("Not enough multi-theme rows for co-occurrence analysis.")